In [62]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything,seed_worker
from utils.data_loading import load_data, load_test_data, feature_names
from models.uvote import UVOTELSTM
import torch.optim as optim
from tqdm.auto import tqdm
# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [63]:
import numpy as np
import torch
from scipy.ndimage import gaussian_filter1d

# ---------------------------------------------------------
def get_lds_weights_from_loader(dataloader, bins=150, sigma=4):
    """
    Extracts all targets from the dataloader and calculates LDS weights.
    """
    print("Calculating LDS weights...")
    all_targets = []
    for _, y in dataloader:
        # y is likely [batch, target_dim] or [batch]
        all_targets.append(y.detach().cpu().numpy())
    
    all_targets = np.concatenate(all_targets).flatten()
    
    # Create histogram
    counts, bin_edges = np.histogram(all_targets, bins=bins)
    
    # Apply Gaussian smoothing to the counts
    smoothed_counts = gaussian_filter1d(counts.astype(float), sigma=sigma)
    
    # Invert and normalize
    weights = 1.0 / (smoothed_counts + 1e-6)
    weights = weights / np.mean(weights)
    
    # Function to map a specific target value to its LDS weight
    def get_weight(val):
        idx = np.digitize(val, bin_edges[:-1]) - 1
        return weights[np.clip(idx, 0, bins-1)]
    
    return get_weight

In [64]:
    seed_everything(42)  # for reproducibility

In [65]:
print("Loading train data...")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path = "../../datasets_challenge/train",  # Path to training data
    train_ratio = 0.98,  # Use 98% for training, 2% for validation
    batchsize = 64,  # Number of samples per batch
    columns_to_standardize = feature_names,  # Option to specify which columns to standardize
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],  # Remove these from features
    sequence_length = 20,  # Number of time steps in each sequence (t-19 to t0)
    threshold = True,  # Enable velocity thresholding
    threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
    threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
    target = "sideSlip",  # The variable we want to predict
    scaling = True,  # Apply standardization to features
    seed = 42,  # Random seed for train/val split
)

Loading train data...
../../datasets_challenge/train\train_part_001.csv
../../datasets_challenge/train\train_part_002.csv
../../datasets_challenge/train\train_part_003.csv
../../datasets_challenge/train\train_part_004.csv
../../datasets_challenge/train\train_part_005.csv
../../datasets_challenge/train\train_part_006.csv
../../datasets_challenge/train\train_part_007.csv
../../datasets_challenge/train\train_part_008.csv
../../datasets_challenge/train\train_part_009.csv
../../datasets_challenge/train\train_part_010.csv
../../datasets_challenge/train\train_part_011.csv
../../datasets_challenge/train\train_part_012.csv
../../datasets_challenge/train\train_part_013.csv
../../datasets_challenge/train\train_part_014.csv
../../datasets_challenge/train\train_part_015.csv
../../datasets_challenge/train\train_part_016.csv
../../datasets_challenge/train\train_part_017.csv
../../datasets_challenge/train\train_part_018.csv
../../datasets_challenge/train\train_part_019.csv
../../datasets_challenge/tra

In [ ]:
# ---------------------------------------------------------
# 3. CUSTOM LOSS & TRAINING FUNCTION
# ---------------------------------------------------------
def train_uvote_lds(model, train_loader, weight_fn, epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            x, y = x.to(device), y.to(device)
            
            # Get LDS weights for this batch based on targets
            # Converting weights to tensor: [batch]
            batch_weights = torch.FloatTensor([weight_fn(val.item()) for val in y]).to(device)
            
            optimizer.zero_grad()
            outputs = model(x) # [experts, batch, 2]
            
            batch_loss = 0
            for i in range(len(model.experts)):
                mu = outputs[i, :, 0]
                log_var = outputs[i, :, 1]
                
                # Negative Log Likelihood Loss
                precision = torch.exp(-log_var)
                mse_part = precision * (y.view(-1) - mu)**2
                loss = (mse_part + log_var) * batch_weights # LDS
                batch_loss += loss.mean()
            
            batch_loss.backward()
            optimizer.step()
            total_loss += batch_loss.item()
            
        print(f"Average Epoch Loss: {total_loss / len(train_loader):.4f}")

In [66]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm
import numpy as np

def train_specialized_uvote(model, train_dataset, val_dataloader, weight_fn, 
                             epochs=200, lr=1e-3, drift_threshold=5.0):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # --- 1. ROBUST LABEL EXTRACTION ---
    print("Extracting labels for Sampler (this may take a moment)...")
    all_labels = []

    # We use the existing loader because it has already handled the pipeline logic
    for _, y in tqdm(train_dataloader, desc="Scanning Labels"):
        # y shape is likely [batch, seq_len] or [batch, 1]
        if y.dim() > 1:
            # Take the last value of the sequence for every item in the batch
            last_values = y[:, -1].cpu().numpy()
        else:
            last_values = y.cpu().numpy()
        all_labels.extend(last_values)

    # Calculate weights for every single sample we found
    sample_weights = [weight_fn(val) for val in all_labels]

    # --- 2. CREATE SAMPLER ---
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # --- 3. RE-CREATE LOADER ---
    train_loader = DataLoader(
        train_dataloader.dataset, 
        batch_size=train_dataloader.batch_size, 
        sampler=sampler,
        worker_init_fn=seed_worker,
        generator=train_dataloader.generator
    )
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # Cosine annealing helps fine-tune the many-shot region at the end
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device).view(-1)
            
            # Get LDS weights for loss scaling
            batch_weights = torch.FloatTensor([weight_fn(val.item()) for val in y]).to(device)
            
            optimizer.zero_grad()
            outputs = model(x) # [experts, batch, 2]
            
            batch_total_loss = 0
            num_experts = len(model.experts)
            
            for i in range(num_experts):
                mu = outputs[i, :, 0]
                log_var = outputs[i, :, 1]
                precision = torch.exp(-log_var)
                
                # --- 2. SPECIALIST LOGIC ---
                # Expert 0: Generalist (Standard weights)
                # Expert 1 & 2: Few-Shot Specialists (Higher weights for high angles)
                expert_boost = batch_weights.clone()
                if i > 0: 
                    # Double the importance of drift samples for specialists
                    expert_boost[y > drift_threshold] *= 2.0 
                
                mse_part = precision * (y - mu)**2
                # We add a 0.05 'Many-shot protection' constant to keep the 0.201 error down
                loss = (mse_part + log_var) * (expert_boost + 0.05)
                batch_total_loss += loss.mean()
            
            batch_total_loss.backward()
            optimizer.step()
            total_loss += batch_total_loss.item()
            
        # --- 3. EVALUATION ---
        metrics = evaluate_metrics(model, val_dataloader, device, drift_threshold)
        scheduler.step()
        
        print(f"\n[Epoch {epoch+1}] "
              f"Many-Shot MAE: {metrics['many_mae']:.4f} | "
              f"Few-Shot MAE: {metrics['few_mae']:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    return model

def evaluate_metrics(model, loader, device, threshold):
    model.eval()
    many_errors = []
    few_errors = []
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device).view(-1)
            outputs = model(x)
            
            # Uncertainty Voting
            means = outputs[:, :, 0]
            precisions = torch.exp(-outputs[:, :, 1])
            voted_pred = (means * precisions).sum(dim=0) / precisions.sum(dim=0)
            
            abs_err = torch.abs(voted_pred - y).cpu().numpy()
            y_np = y.cpu().numpy()
            
            many_errors.extend(abs_err[y_np <= threshold])
            few_errors.extend(abs_err[y_np > threshold])
            
    return {
        "many_mae": np.mean(many_errors) if many_errors else 0,
        "few_mae": np.mean(few_errors) if few_errors else 0
    }

In [67]:
# ---------------------------------------------------------
# 4. EXECUTION
# ---------------------------------------------------------

# 1. Calculate the weight function using your existing train_dataloader
weight_func = get_lds_weights_from_loader(train_dataloader)

# 2. Initialize Model
# input_dim should be number of features after dropping columns
input_dim = 22
model = UVOTELSTM(input_dim=input_dim, num_experts=3)

# 3. Train
train_specialized_uvote(model, train_dataloader, val_dataloader, weight_func, 
                             epochs=200, lr=1e-3, drift_threshold=5.0)

Calculating LDS weights...
Extracting labels for Sampler (this may take a moment)...


Scanning Labels:   0%|          | 0/2100 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/2100 [00:00<?, ?it/s]


[Epoch 1] Many-Shot MAE: 1.4554 | Few-Shot MAE: 2.9373 | LR: 0.001000


Epoch 2:   0%|          | 0/2100 [00:00<?, ?it/s]


[Epoch 2] Many-Shot MAE: 0.9634 | Few-Shot MAE: 1.0106 | LR: 0.001000


... (393 outputs left out)


[Epoch 199] Many-Shot MAE: 0.2351 | Few-Shot MAE: 0.0966 | LR: 0.000000


Epoch 200:   0%|          | 0/2100 [00:00<?, ?it/s]


[Epoch 200] Many-Shot MAE: 0.2351 | Few-Shot MAE: 0.0966 | LR: 0.000000


UVOTELSTM(
  (lstm): LSTM(22, 64, batch_first=True)
  (experts): ModuleList(
    (0-2): 3 x Linear(in_features=64, out_features=2, bias=True)
  )
)

In [32]:
# Validation phase
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
running_val_loss = 0
running_val_mae = 0

# Track Many-Shot and Few-Shot specifically
many_errors = []
few_errors = []
drift_threshold = 5.0 # Define what counts as a high-angle drift

with torch.no_grad():
    for val_step, (x, y) in enumerate(val_dataloader):
        X_val, y_val = x.to(device), y.to(device).view(-1) # Flatten target for easier math
        
        # 1. Forward pass returns [experts, batch, 2]
        outputs = model(X_val)
        
        # 2. UVOTE Inference: Calculate the weighted average based on confidence
        means = outputs[:, :, 0]
        precisions = torch.exp(-outputs[:, :, 1]) # Inverse of uncertainty
        voted_pred = (means * precisions).sum(dim=0) / precisions.sum(dim=0)
        
        # 3. Calculate Loss (Using NLL for validation consistency)
        # Note: You can also use standard MSE/MAE here for easier tracking
        batch_loss = 0
        for i in range(len(model.experts)):
            mu = outputs[i, :, 0]
            log_var = outputs[i, :, 1]
            p = torch.exp(-log_var)
            # We don't apply LDS weights during validation evaluation
            batch_loss += (p * (y_val - mu)**2 + log_var).mean()
        
        running_val_loss += (batch_loss / len(model.experts)).item()
        
        # 4. Calculate MAE based on the VOTED prediction
        abs_errors = torch.abs(voted_pred - y_val)
        running_val_mae += abs_errors.mean().item()
        
        # 5. Split errors for analysis (Optional but highly recommended)
        y_cpu = y_val.cpu().numpy()
        err_cpu = abs_errors.cpu().numpy()
        many_errors.extend(err_cpu[y_cpu <= drift_threshold])
        few_errors.extend(err_cpu[y_cpu > drift_threshold])

# Finalizing Metrics
avg_val_loss = running_val_loss / (val_step + 1)
avg_val_mae = running_val_mae / (val_step + 1)

# Analyze Many-Shot vs Few-Shot
avg_few_mae = np.mean(few_errors) if few_errors else 0
avg_many_mae = np.mean(many_errors) if many_errors else 0

In [68]:
# 1. Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("\nLoading test data...")
test_dataloader = load_test_data(
    test_path = "../../datasets_challenge/test",
    batchsize = 64,
    pipeline = pipeline, 
    load_anonymized_data=True
)

# 2. Generate predictions using Uncertainty Voting
model.eval()
test_preds = []

print("Running UVOTE Inference on Test Set...")
with torch.no_grad():
    for x, y in test_dataloader:
        X = x.to(device)

        # UVOTE Forward Pass: returns [experts, batch, 2]
        outputs = model(X)
        
        # Extract Means and Precisions (inverse of uncertainty)
        means = outputs[:, :, 0]
        # We use exp(-log_var) to get precision
        precisions = torch.exp(-outputs[:, :, 1]) 
        
        # PERFORM THE VOTE: Weighted average by confidence
        # This is the "Magic" that improves few-shot errors
        voted_pred = (means * precisions).sum(dim=0) / precisions.sum(dim=0)
        
        # Store predictions
        test_preds.append(voted_pred.cpu().numpy())

# 3. Concatenate and Save
test_preds = np.concatenate(test_preds)

print("Shape of Predictions:", test_preds.shape)

# Save predictions in the required format
y_pred = pd.DataFrame({
    "id": range(len(test_preds)),
    "predicted_sideslip": test_preds
})

# Reorder columns to ensure "id" is first if required by competition
y_pred = y_pred[["id", "predicted_sideslip"]]

# Save to CSV
y_pred.to_csv("predictions_uvote3.csv", index=False)

print(f"✓ Predictions saved to 'predictions_uvote3.csv'")
print(f"✓ Total samples: {len(test_preds)}")


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv
Running UVOTE Inference on Test Set...
Shape of Predictions: (49328,)
✓ Predictions saved to 'predictions_uvote3.csv'
✓ Total samples: 49328
